In [1]:
import pandas as pd 
import numpy as np

In [2]:
data = pd.read_csv('../data/rxavierlabeled_articles.csv', index_col='id')
data

,text,authenticity_score,sensationalism_score,political_bias_score,toxicity_score,confirmation_bias_score,short_term_utility_score
id,,,,,,,
1,"NEW YORK, NY — Mayor-elect Zohran Mamdani anno...",2,7,5,9,3,8
2,Walmart is proving to be America’s antidote to...,7,5,2,1,5,3
3,LAS VEGAS—Shaking his head in frustration afte...,1,9,2,5,8,10
4,Nov 20 (Reuters) - Studies from Novo Nordisk (...,9,4,1,2,6,2
5,The Buffalo Bills and Houston Texans will meet...,8,2,1,1,6,5
6,"For the first time in his second term, Preside...",7,6,8,4,2,6
7,While hosting Saudi Crown Prince Mohammed bin ...,9,7,1,8,5,8
8,SAN DIEGO — San Diego homeowners looking to se...,7,2,1,1,6,4
9,The Democrat is accused of stealing Federal Em...,8,7,7,6,8,3


# Function and Prompt Definitions

In [3]:
import os
from openai import OpenAI
import json
import re
import yaml

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config["nautilus"]["api"]

In [31]:
def baseline_prompt(article_text, pred_vec=None):
    return f"""
You are an AI tasked with scoring news articles. 

For the following article, assign numeric scores from 1 to 10 for each of these six factors: 

1. Authenticity
2. Sensationalism
3. Political Bias
4. Spam
5. Confirmation Bias
6. Short-Term Utility

Do not provide explanations, examples, or comments. Only output a Python dictionary with these exact keys and numeric values.  

Article:
{article_text}
"""

def refined_prompt(article_text, pred_vec=None):
    return f"""
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION
The feature vector contains the following predictive model outputs and auxiliary measures:
0-5: Probabilities for truthfulness classes from our custom BERT-based model:
     0 = False, 1 = Half True, 2 = Mostly True, 3 = True, 4 = Barely True, 5 = Pants on Fire
7: Count of numeric/statistical entities detected in the text
8: Count of conservative bigram matches in the text
9: Count of liberal bigram matches in the text
10: Emotional intensity score (absolute VADER compound score)
11: Spam likelihood score (0–1, probability of being spam)

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. Spam
- Definition: Determine whether a piece of content qualifies as spam, and assess whether the spam contains or contributes to disinformation.
- Scoring Recipe (1–10): Score based on how strongly the content exhibits spam characteristics.
- Output: score + example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Spam", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

ARTICLE TEXT:
\"\"\"
{article_text}
\"\"\"

PREDICTIVE MODEL FEATURE VECTOR:
[{pred_vec}]
"""

def final_prompt(article_text, pred_vec):
    return f"""You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.

Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

FACTUALITY FACTORS (6 TOTAL):

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. Spam
- Definition: Determine whether a piece of content qualifies as spam, and assess whether the spam contains or contributes to disinformation.
- Scoring Recipe (1–10): Score based on how strongly the content exhibits spam characteristics.
- Output: score + example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

VERACITY LABEL (REQUIRED)
- Output a final factuality classification: pants-fire, false, barely-true, half-true, mostly-true, true

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Spam", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

REASONING FORMAT
- Provide concise, structured rationale per factor: key textual evidence, weighting of predictive scores, numeric reasoning.  
- DO NOT reveal internal chain-of-thought or hidden reasoning.

EXAMPLES
Here are some examples of the expected output format and reasoning structure:
    - Example 1:
    input: 
    {{
        "article": "A new study shows that drinking green tea daily reduces the risk of heart disease by 30%. The study surveyed 10,000 adults over 5 years and was published in the Journal of Cardiology.", 
        "source": "Health Daily News"
    }}
    result:
    {{
        "veracity_label": "Mostly True",
        "explanation_text": "The article cites a study published in a reputable journal with a large sample size and multi-year scope, which strongly supports its credibility. While the claim that green tea reduces heart disease risk by 30% is plausible, the phrasing slightly overstates certainty by not discussing limitations or confidence intervals. There is no political agenda or hostile language, and the article is primarily informational rather than engagement-driven.",
        "factor_scores": [
            {{"factor": "Authenticity", "score": 9, "reasoning": "Published in a reputable journal with a large sample size and clear methodology."}},
            {{"factor": "Sensationalism", "score": 3, "reasoning": "Slightly bold phrasing but largely factual and restrained."}},
            {{"factor": "Political Bias", "score": 1, "reasoning": "No political framing or agenda detected."}},
            {{"factor": "Spam", "score": 1, "reasoning": "No deceptive, repetitive, or manipulative content."}},
            {{"factor": "Confirmation Bias", "score": 2, "reasoning": "Emphasizes positive findings with limited discussion of caveats."}},
            {{"factor": "Short-term Utility", "score": 2, "reasoning": "Primarily informational with mild attention-grabbing appeal."}}
        ]
    }}
    - Example 2:
    input:
    {{
        "article": "Politician X is the worst leader in history! Everything they touch fails, and the economy is collapsing under their rule.", 
        "source": "Partisan Weekly"}}
    result:
    {{
        "veracity_label": "False",
        "explanation_text": "The article makes sweeping negative claims about a political figure without citing evidence or verifiable data. The language is highly emotional, attacking the individual rather than evaluating specific policies or outcomes. Strong political bias and selective presentation of information undermine credibility, suggesting the content is designed more to provoke outrage than inform.",
        "factor_scores": [
            {{"factor": "Authenticity", "score": 2, "reasoning": "Claims are vague and unsupported by evidence or sources."}},
            {{"factor": "Sensationalism", "score": 9, "reasoning": "Uses hyperbolic and extreme language to provoke emotion."}},
            {{"factor": "Political Bias", "score": 10, "reasoning": "Explicit partisan framing targeting a political figure."}},
            {{"factor": "Spam", "score": 8, "reasoning": "Aggressive rhetoric and personal attacks resemble engagement-driven spam content."}},
            {{"factor": "Confirmation Bias", "score": 8, "reasoning": "Only negative information is presented; counterpoints are ignored."}},
            {{"factor": "Short-term Utility", "score": 7, "reasoning": "Clearly designed to trigger outrage and maximize clicks."}}
        ]
    }}
    - Example 3:
    input:
    {{
        "article": "Local bakery wins award for best chocolate cake. The contest included 50 bakeries, and judges highlighted creativity and flavor balance.", 
        "source": "Town Gazette"
    }}
    result:
    {{
        "veracity_label": "True",
        "explanation_text": "The article reports on a verifiable local contest with identifiable participants and judging criteria. The tone is neutral and factual, with no exaggeration or bias. The content is a straightforward human-interest story with no indication of manipulation or selective framing.",
        "factor_scores": [
            {{"factor": "Authenticity", "score": 8, "reasoning": "Contest details and judging process are verifiable and clearly described."}},
            {{"factor": "Sensationalism", "score": 2, "reasoning": "Positive tone without exaggeration or emotional manipulation."}},
            {{"factor": "Political Bias", "score": 1, "reasoning": "No political content or implications."}},
            {{"factor": "Spam", "score": 1, "reasoning": "No misleading or manipulative characteristics."}},
            {{"factor": "Confirmation Bias", "score": 1, "reasoning": "Balanced reporting with no selective omission."}},
            {{"factor": "Short-term Utility", "score": 3, "reasoning": "Lightly engaging but not engineered for virality."}}
        ]
    }}

ARTICLE TEXT:
[{article_text}]

PREDICTED VECTOR:
[{pred_vec}]
"""


def run_genai(article_text, feature_vector=None, prompt=None):
    prompt = prompt(article_text, feature_vector)
    client = OpenAI(
        api_key = api_key, 
        base_url = "https://ellm.nrp-nautilus.io/v1"
    )

    completion = client.chat.completions.create(
        model="gemma3",
        messages=[
          
            {"role": "system", "content": "You are a helpful assistant that outputs STRICT JSON only."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1 
    )
    return completion.choices[0].message.content

# Prompt evaluation

## Naive Prompt

In [32]:
responses_naive = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    response = run_genai(article_text=txt, prompt=baseline_prompt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_naive.append(response_json)

In [51]:
naive_df = pd.DataFrame(responses_naive)
naive_df

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
0,1,9,7,1,4,2
1,8,3,4,1,5,7
2,8,6,1,1,2,4
3,9,3,2,1,4,7
4,10,2,1,1,3,8
5,8,4,6,1,5,7
6,8,6,7,1,5,9
7,9,2,3,1,4,8
8,9,5,6,1,5,8
9,8,4,6,1,7,7


## Refined Prompt: Adding scoring recipes

In [7]:
responses_refined = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    response = run_genai(article_text=txt, prompt=refined_prompt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_refined.append(response_json)

df_refined_inter = pd.DataFrame(responses_refined)
df_refined_inter

,veracity_label,explanation_text,factor_scores
0,Pants on Fire,This article is demonstrably false and satiric...,"[{'factor': 'Authenticity', 'score': 1, 'reaso..."
1,Mostly True,The article reports on Walmart's financial per...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
2,True,This article is a humorous piece of fictional ...,"[{'factor': 'Authenticity', 'score': 2, 'reaso..."
3,Mostly True,The article reports on upcoming studies from N...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
4,True,This article presents factual information abou...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
5,Mostly True,The article primarily reports on a shift in Pr...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
6,Mostly True,The article reports a direct quote from Presid...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
7,Mostly True,The article reports on trends in the San Diego...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
8,Mostly True,The article reports on allegations against a C...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
9,Mostly True,The article reports on concerns raised by US l...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."


In [52]:
df_list = []
for i in df_refined_inter['factor_scores']:
    df_temp = pd.DataFrame(i,)
    scores_wide = (
    df_temp
        .assign(_idx=0)
        .pivot(index="_idx", columns="factor", values="score")
        .reset_index()
        .drop(columns="_idx")
    )
    df_list.append(scores_wide)
df_refined = pd.concat(df_list)
df_refined.index = range(10)
df_refined

factor,Authenticity,Confirmation Bias,Political Bias,Sensationalism,Short-term Utility,Spam
0,1,2,6,10,7,2
1,8,2,3,2,2,1
2,2,1,1,6,3,1
3,9,2,1,2,2,1
4,9,1,1,2,3,1
5,8,3,6,4,2,1
6,9,5,7,6,3,1
7,9,2,1,2,3,1
8,9,2,4,3,2,1
9,8,3,6,3,2,1


## Final Prompt: In-context learning

In [9]:
responses_final = []
for i in range(1,data.shape[0]+1):
    txt = data.loc[i, 'text']
    response = run_genai(article_text=txt, prompt=final_prompt)
    clean_str = re.sub(r"```json|```", "", response).strip()
    if "{" in clean_str:
        start = clean_str.find("{")
        end = clean_str.rfind("}") + 1
        clean_str = clean_str[start:end]
    response_json = json.loads(clean_str)
    responses_final.append(response_json)

df_final_inter = pd.DataFrame(responses_final)
df_final_inter

,veracity_label,explanation_text,factor_scores
0,Pants on Fire,The article presents a highly improbable and s...,"[{'factor': 'Authenticity', 'score': 1, 'reaso..."
1,Mostly True,The article presents factual information about...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
2,True,"The article reports a peculiar, yet verifiable...","[{'factor': 'Authenticity', 'score': 8, 'reaso..."
3,Mostly True,The article reports on upcoming clinical trial...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
4,True,The article presents factual information about...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
5,Mostly True,The article presents a factual account of a sh...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
6,Mostly False,The article reports President Trump's call to ...,"[{'factor': 'Authenticity', 'score': 6, 'reaso..."
7,True,The article presents factual data regarding th...,"[{'factor': 'Authenticity', 'score': 9, 'reaso..."
8,Mostly True,The article reports on ongoing legal allegatio...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."
9,Mostly True,The article accurately reports on concerns rai...,"[{'factor': 'Authenticity', 'score': 8, 'reaso..."


In [53]:
df_list = []
for json_list in df_final_inter['factor_scores']:
    df_temp = pd.DataFrame(json_list,)
    scores_wide = (
    df_temp
        .assign(_idx=0)
        .pivot(index="_idx", columns="factor", values="score")
        .reset_index()
        .drop(columns="_idx")
    )
    df_list.append(scores_wide)
df_final = pd.concat(df_list)
df_final.index = range(10)
df_final

factor,Authenticity,Confirmation Bias,Political Bias,Sensationalism,Short-term Utility,Spam
0,1,1,7,10,8,9
1,8,2,1,3,3,1
2,8,1,1,3,4,1
3,8,2,1,3,3,1
4,9,1,1,2,3,1
5,8,3,6,4,3,1
6,6,7,8,7,5,4
7,9,2,1,2,3,1
8,8,3,2,3,4,1
9,8,5,6,4,3,1


# Post-processing LLM Outputs

In [77]:
naive_df.index.name = 'id'
naive_df.columns.name = None
naive_df.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# naive_df.index += 1
naive_df = naive_df[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(naive_df)

df_refined.index.name = 'id'
df_refined.columns.name = None
df_refined.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# df_refined.index += 1
df_refined = df_refined[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(df_refined)

df_final.index.name = 'id'
df_final.columns.name = None
df_final.rename(columns={'Short-term Utility':'Short-Term Utility'}, inplace=True)
# df_final.index += 1
df_final = df_final[['Authenticity','Sensationalism','Political Bias','Spam','Confirmation Bias','Short-Term Utility']]
display(df_final)

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
1,1,9,7,1,4,2
2,8,3,4,1,5,7
3,8,6,1,1,2,4
4,9,3,2,1,4,7
5,10,2,1,1,3,8
6,8,4,6,1,5,7
7,8,6,7,1,5,9
8,9,2,3,1,4,8
9,9,5,6,1,5,8


,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
1,1,10,6,2,2,7
2,8,2,3,1,2,2
3,2,6,1,1,1,3
4,9,2,1,1,2,2
5,9,2,1,1,1,3
6,8,4,6,1,3,2
7,9,6,7,1,5,3
8,9,2,1,1,2,3
9,9,3,4,1,2,2


,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
1,1,10,7,9,1,8
2,8,3,1,1,2,3
3,8,3,1,1,1,4
4,8,3,1,1,2,3
5,9,2,1,1,1,3
6,8,4,6,1,3,3
7,6,7,8,4,7,5
8,9,2,1,1,2,3
9,8,3,2,1,3,4


In [78]:
ground_truth = data.drop(columns='text')
ground_truth.rename(columns={'short_term_utility_score':'Short-term Utility','toxicity_score':'Spam'}, inplace=True)
ground_truth.rename(columns=lambda x: x.replace('_score','').replace('_',' ').title(), inplace=True)
ground_truth

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
id,,,,,,
1,2,7,5,9,3,8
2,7,5,2,1,5,3
3,1,9,2,5,8,10
4,9,4,1,2,6,2
5,8,2,1,1,6,5
6,7,6,8,4,2,6
7,9,7,1,8,5,8
8,7,2,1,1,6,4
9,8,7,7,6,8,3


# Prompt Results

In [84]:
naive_results = (naive_df == ground_truth).mean()
naive_results

Authenticity          0.1
Sensationalism        0.2
Political Bias        0.1
Spam                  0.3
Confirmation Bias     0.2
Short-Term Utility    0.0
dtype: float64

In [85]:
refined_results = (df_refined == ground_truth).mean()
refined_results

Authenticity          0.2
Sensationalism        0.2
Political Bias        0.3
Spam                  0.3
Confirmation Bias     0.2
Short-Term Utility    0.1
dtype: float64

In [86]:
final_results = (df_final == ground_truth).mean()
final_results

Authenticity          0.1
Sensationalism        0.3
Political Bias        0.3
Spam                  0.4
Confirmation Bias     0.0
Short-Term Utility    0.2
dtype: float64

In [95]:
result_df = pd.concat([naive_results,refined_results,final_results], axis=1).T
result_df['Prompt'] = ['Naive','Refined','Final']
result_df.set_index('Prompt', inplace=True)
result_df

,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
Prompt,,,,,,
Naive,0.1,0.2,0.1,0.3,0.2,0.0
Refined,0.2,0.2,0.3,0.3,0.2,0.1
Final,0.1,0.3,0.3,0.4,0.0,0.2


In [96]:
result_df.to_csv('../data/prompting_results.csv')

In [97]:
result_df = pd.read_csv('../data/prompting_results.csv')
result_df

,Prompt,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility
0,Naive,0.1,0.2,0.1,0.3,0.2,0.0
1,Refined,0.2,0.2,0.3,0.3,0.2,0.1
2,Final,0.1,0.3,0.3,0.4,0.0,0.2


In [98]:
descs = np.array(["Standard prompt for our task at hand, giving minimal details for the LLM to work with",
         "Standard prompt + scoring recipes for the LLM to use and evaluate",
         "Standard prompt + scoring recipes and example outputs (in-context learning) for best possible results"])

prompts = np.array([baseline_prompt('<article>'),
                    refined_prompt('<article>','<predictive model output>'),
                    final_prompt('<article>','<predictive model output>')])

In [ ]:
result_df['description'] = descs
result_df['full_prompt'] = prompts

result_df

,Prompt,Authenticity,Sensationalism,Political Bias,Spam,Confirmation Bias,Short-Term Utility,description,full_propmt,full_prompt
0,Naive,0.1,0.2,0.1,0.3,0.2,0.0,"Standard prompt for our task at hand, giving m...",\nYou are an AI tasked with scoring news artic...,\nYou are an AI tasked with scoring news artic...
1,Refined,0.2,0.2,0.3,0.3,0.2,0.1,Standard prompt + scoring recipes for the LLM ...,\nYou are an AI assistant assigned to evaluate...,\nYou are an AI assistant assigned to evaluate...
2,Final,0.1,0.3,0.3,0.4,0.0,0.2,Standard prompt + scoring recipes and example ...,You are an AI assistant assigned to evaluate t...,You are an AI assistant assigned to evaluate t...


In [ ]:
result_df.to_csv('../data/prompting_results.csv')

In [103]:
(df_final == df_refined).mean()

Authenticity          0.6
Sensationalism        0.5
Political Bias        0.6
Spam                  0.8
Confirmation Bias     0.6
Short-Term Utility    0.2
dtype: float64

In [104]:
(df_final == naive_df).mean()

Authenticity          0.6
Sensationalism        0.6
Political Bias        0.5
Spam                  0.8
Confirmation Bias     0.0
Short-Term Utility    0.1
dtype: float64